In [ ]:
# !pip install "unstructured[pdf]"
# !brew install poppler
# !brew install tesseract
# !pip install -qU langchain-unstructured
# !pip install langchain_qdrant
# from langchain_community.document_loaders import DirectoryLoader

In [ ]:
import os
import glob
import logging
from llmops.document_loader import DocumentLoader, UnstructuredPDFMethod
from pprint import pprint
import warnings
warnings.filterwarnings("ignore")

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_ollama import OllamaEmbeddings


from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.llms import OllamaLLM

from langchain import hub
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import fitz
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image


def plot_pdf_with_boxes(pdf_page, segments):
    pix = pdf_page.get_pixmap()
    pil_image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(pil_image)
    categories = set()
    category_to_color = {
        "Title": "orchid",
        "Image": "forestgreen",
        "Table": "tomato",
    }
    for segment in segments:
        points = segment["coordinates"]["points"]
        layout_width = segment["coordinates"]["layout_width"]
        layout_height = segment["coordinates"]["layout_height"]
        scaled_points = [
            (x * pix.width / layout_width, y * pix.height / layout_height)
            for x, y in points
        ]
        box_color = category_to_color.get(segment["category"], "deepskyblue")
        categories.add(segment["category"])
        rect = patches.Polygon(
            scaled_points, linewidth=1, edgecolor=box_color, facecolor="none"
        )
        ax.add_patch(rect)

    # Make legend
    legend_handles = [patches.Patch(color="deepskyblue", label="Text")]
    for category in ["Title", "Image", "Table"]:
        if category in categories:
            legend_handles.append(
                patches.Patch(color=category_to_color[category], label=category)
            )
    ax.axis("off")
    ax.legend(handles=legend_handles, loc="upper right")
    plt.tight_layout()
    plt.show()


def render_page(doc_list: list, page_number: int, print_text=True) -> None:
    pdf_page = fitz.open(file_paths[0]).load_page(page_number - 1)
    page_docs = [
        doc for doc in doc_list if doc.metadata.get("page_number") == page_number
    ]
    segments = [doc.metadata for doc in page_docs]
    plot_pdf_with_boxes(pdf_page, segments)
    if print_text:
        for doc in page_docs:
            print(f"{doc.page_content}\n")

## File Paths

In [ ]:
# Creating file paths
DATA_DIR = "./DataFiles/PDFs"
file_paths = glob.glob(os.path.join(DATA_DIR, "*.pdf"))
print(*file_paths, sep="\n")

## Document Loading

In [ ]:
# Instantiate document loader and set parsing strategy
loader = DocumentLoader(parsing_method=UnstructuredPDFMethod())

# Read pdf files
documents = []
for file in file_paths:
    documents += loader.get_document(file_path=file)

pprint(f"Total {len(documents)} pages were read from {len(file_paths)} files.")
# pprint(documents[0])

In [ ]:
# render_page(documents, 4)

In [ ]:
# # Splitting the documents into chunks
# splitter = RecursiveCharacterTextSplitter(
#                                         chunk_size=1000,
#                                         chunk_overlap=200,
#                                         add_start_index=True
#                                         )

# documents_split = splitter.split_documents(documents)

# print(f"Total {len(documents_split)} splits were created from {len(documents)} pages.")

In [ ]:
# documents_cat = [x.metadata['category'] for x in documents]
# print(documents_cat)

# unique_categories = set(documents_cat)
# print(unique_categories)

# Image category documents
# image_docs = [x for x in documents if x.metadata['category'] == 'Image']
# image_docs[0].page_content

# Documents without category "Image"
documents_filtered = [x for x in documents if x.metadata['category'] != 'Image']
print(f"Total {len(documents_filtered)} documents without category 'Image'.")

## Vector Database

In [ ]:
# Initializing the Embedding model and Vector Store
embedding = OllamaEmbeddings(model="llama3.2")

client = QdrantClient(":memory:")

client.create_collection(
                        collection_name="demo_collection",
                        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
                        )


vector_store = QdrantVectorStore(
                                client=client,
                                collection_name="demo_collection",
                                embedding=embedding,
                                )


# Add the documents to the Vector Store
vector_store.add_documents(documents_filtered) #changed from documents_split to documents

# Setting up the retrievar
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 6})

## Prompt

In [ ]:
# Using a predefined prompt
# prompt = hub.pull("rlm/rag-prompt")

system_message = """
You are an assitant who helps in answering questions based on the context provided to you. Provide the paper title for upon which you base your response.
Answer with 'I do not know or I am not sure' if you cannot find answers in the provided context.
"""

human_message = """Based on the following context.
Context: {context}.
---
Answer the question : {question}.

In your answer qoute the provided content metadata at the end of the answer.
Answer : """

prompt = ChatPromptTemplate(
    [
        ("system", system_message),
        ("human", human_message)

    ]
)

In [ ]:
# parser = StrOutputParser()

# chain = prompt 

# response = chain.invoke({"question" : "my question", "context" : "my provided context"})
# print(parser.parse(response).to_messages()[1].content)

In [ ]:
# example_messages = prompt.invoke({
#                                 "context": "filler context", 
#                                 "question": "filler question"
#                                 }).to_messages()

# print(example_messages)

## LLM chain creation

In [ ]:
model_name = "llama3.2"
llm = OllamaLLM(model=model_name)

In [ ]:
# documents[0].metadata

In [ ]:
# Function to combine the retrieved documents
def format_docs(docs):
    # print(len(docs))
    context_string = ""
    for doc in docs:
        context_string += f"""Content:- \n{doc.page_content}\n---\nMetadata:-\nFile name : {doc.metadata["filename"]} | Page number : {doc.metadata["page_number"]} | Category : {doc.metadata["category"]}\n---\n\n"""
    
    # print("="*100)
    # print(context_string)
    # print("="*100)
    
    return context_string
    # return "\n\n".join(f"{doc.page_content}" for doc in docs)


rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
            )

## QA Examples

In [ ]:
logging.getLogger().setLevel(logging.CRITICAL)

In [ ]:
for chunk in rag_chain.stream("Deep learning methods for homography estimation."):
    print(chunk, end="", flush=True)

In [ ]:
for chunk in rag_chain.stream("Which images are availble in the provided paper?"):
    print(chunk, end="", flush=True)

In [ ]:
for chunk in rag_chain.stream("The relationship between the three types of layout data structure"):
    print(chunk, end="", flush=True)

In [ ]:
for chunk in rag_chain.stream("Which figures are discussed in the provided paper?"):
    print(chunk, end="", flush=True)

In [ ]:
for chunk in rag_chain.stream("List the key findings in conclusion."):
    print(chunk, end="", flush=True)

In [ ]:
for chunk in rag_chain.stream("What are the mentioned approachs in the papers?"):
    print(chunk, end="", flush=True)

In [ ]:
for chunk in rag_chain.stream("Explain in detail about 'The Core LayoutParser Library'"):
    print(chunk, end="", flush=True)

In [ ]:
for chunk in rag_chain.stream("How can LayoutParser be used for document analysis?"):
    print(chunk, end="", flush=True)

In [ ]:
for chunk in rag_chain.stream("Discuss the use cases of LayoutParser."):
    print(chunk, end="", flush=True)